# VoiceBridge — Offline Multilingual Clinical Triage AI
### Gemma 4 Good Hackathon 2026 · Digital Equity Track

**VoiceBridge** is an offline-capable multilingual clinical intake AI built on [Gemma 4 E4B](https://huggingface.co/google/gemma-4-e4b-it),
fine-tuned with LoRA on 500 synthetic SATS/WHO ETAT triage cases.
It assigns one of five triage levels (RED → BLUE) from a spoken or written patient intake,
works in five languages (English, Swahili, Tagalog, Bengali, Hausa), and runs fully offline
on a Raspberry Pi 5 8 GB via llama.cpp Q4\_K\_M quantisation, no cloud required.

| Resource | Link |
|---|---|
| HuggingFace model | https://huggingface.co/OminousDude/voicebridge-gemma4 |
| GitHub repository | https://github.com/MaximG6/Gemma4Kaggle |

---
**Benchmark highlights (20-case evaluation, Q4\_K\_M GGUF):**
- Fine-tuned model: **95% exact match**, **100% safe escalation**
- Base model: 65% exact match, 90% safe escalation  (+30 pp delta)
- Mean latency: 6.2 s on RTX 5090 · target < 8 s on Pi 5 ✓

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
import subprocess, sys

print("Installing Python dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

print("Installing system packages...")
subprocess.run(
    "apt-get install -y cmake build-essential git > /dev/null 2>&1",
    shell=True, check=True
)
print("Dependencies ready.")

In [ ]:
# ── Cell 3: Build llama.cpp and download model ────────────────────────────
import os, subprocess, sys
from pathlib import Path

LLAMA_DIR  = Path("/kaggle/working/llama.cpp")
LLAMA_CLI  = LLAMA_DIR / "build" / "bin" / "llama-cli"
MODEL_PATH = Path("/kaggle/working/voicebridge-finetuned-q4km.gguf")

# ── Build llama.cpp ───────────────────────────────────────────────────────
if LLAMA_DIR.exists():
    print("llama.cpp already built — skipping.")
else:
    print("Cloning llama.cpp...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp",
         str(LLAMA_DIR)],
        check=True
    )
    build_dir = LLAMA_DIR / "build"
    build_dir.mkdir(parents=True, exist_ok=True)
    print("Running cmake...")
    subprocess.run(
        ["cmake", "..", "-DGGML_NATIVE=OFF"],
        cwd=str(build_dir), check=True
    )
    print("Building llama.cpp (this takes ~3 minutes)...")
    subprocess.run(
        ["cmake", "--build", ".", "-j4"],
        cwd=str(build_dir), check=True
    )
    print(f"Build complete. Binary at: {LLAMA_CLI}")

# ── Download GGUF ─────────────────────────────────────────────────────────
os.environ["HF_HUB_DISABLE_XET"] = "1"

if MODEL_PATH.exists():
    size_gb = MODEL_PATH.stat().st_size / 1e9
    print(f"Model already downloaded ({size_gb:.2f} GB) — skipping.")
else:
    print("Downloading voicebridge-finetuned-q4km.gguf from HuggingFace...")
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id="OminousDude/voicebridge-gemma4",
        filename="voicebridge-finetuned-q4km.gguf",
        local_dir="/kaggle/working",
    )
    size_gb = MODEL_PATH.stat().st_size / 1e9
    print(f"Download complete: {size_gb:.2f} GB")

In [ ]:
# ── Cell 4: Inference engine ──────────────────────────────────────────────
from __future__ import annotations
import json, os, re, shlex, subprocess, time
from pathlib import Path
from typing import Optional

LLAMA_CLI  = "/kaggle/working/llama.cpp/build/bin/llama-cli"
MODEL_PATH = "/kaggle/working/voicebridge-finetuned-q4km.gguf"

# System prompt — read from repo if available, otherwise embedded
_PROMPT_FILE = Path("/kaggle/working/Gemma4Kaggle/voicebridge/prompts/triage_system.txt")
if _PROMPT_FILE.exists():
    SYSTEM_PROMPT = _PROMPT_FILE.read_text(encoding="utf-8")
else:
    SYSTEM_PROMPT = """You are a clinical triage assistant (SATS 2023 / WHO ETAT). Language: {lang_name}.
Output ONLY a JSON object with these exact fields:
  triage_level        \u2014 lowercase only: red, orange, yellow, green, or blue
  primary_complaint   \u2014 exactly one sentence, clinical diagnosis only
  red_flag_indicators \u2014 JSON array of strings always, use [] if none
  recommended_action  \u2014 maximum 2 sentences, specific and actionable only
  confidence_score    \u2014 float between 0.0 and 1.0 only, never an integer
All field values must be in English regardless of input language.

Follow this decision tree in order \u2014 stop at the first match:

BLUE   -> confirmed death (rigor mortis + fixed pupils + cold body + no vital signs)
RED    -> ANY: no breathing/pulse | active seizure >5min | AVPU=U | SpO2<85 | SBP<80 with HR>130 | eclampsia
ORANGE -> ANY: suspected MI with stable BP | acute stroke | severe sepsis | SpO2 85-92 | AVPU=V | glucose <3
YELLOW -> ANY: moderate pain stable vitals | fever in child alert | head injury GCS>13 | stable haematemesis
GREEN  -> none of the above, patient alert, vitals normal

KEY RULE: If the patient is alert and talking and SBP is above 90 \u2014 do NOT assign red. Use orange at most.
Only include red_flag_indicators that are explicitly stated in the report. Do not infer or assume missing vitals."""

LANG_NAMES: dict[str, str] = {
    "en": "English", "sw": "Swahili", "tl": "Tagalog",
    "ha": "Hausa",   "bn": "Bengali", "am": "Amharic",
    "hi": "Hindi",   "fr": "French",
}

_LEVELS = ["red", "orange", "yellow", "green", "blue"]


def build_prompt(text: str, lang: str) -> str:
    sp = SYSTEM_PROMPT.format(lang_name=LANG_NAMES.get(lang, "English"))
    return (
        f"<start_of_turn>system\n{sp}<end_of_turn>\n"
        f"<start_of_turn>user\n{text}<end_of_turn>\n"
        f"<start_of_turn>model\n{{"
    )


def _normalise(raw: str) -> Optional[str]:
    r = raw.lower().strip()
    return r if r in _LEVELS else None


def run_triage(text: str, lang: str = "en") -> dict:
    prompt   = build_prompt(text, lang)
    pid      = os.getpid()
    tmp_path = Path(f"/tmp/vb_{pid}_{int(time.time()*1000)}.typescript")

    t0 = time.time()
    try:
        cmd_str = " ".join(shlex.quote(c) for c in [
            LLAMA_CLI, "-m", MODEL_PATH, "-p", prompt,
            "-n", "600", "--threads", "4",
            "--temp", "0.1", "--repeat-penalty", "1.3",
            "-ngl", "0", "--single-turn", "--log-disable",
        ])
        subprocess.run(
            ["script", "-q", "-c", cmd_str, str(tmp_path)],
            stdin=subprocess.DEVNULL,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=600, check=False,
        )
    except subprocess.TimeoutExpired:
        tmp_path.unlink(missing_ok=True)
        return {"triage_level": None, "latency_s": 600.0, "error": "TIMEOUT"}
    except Exception as exc:
        tmp_path.unlink(missing_ok=True)
        return {"triage_level": None, "latency_s": 0.0, "error": str(exc)}

    latency  = time.time() - t0
    raw_full = tmp_path.read_text(errors="replace").strip() if tmp_path.exists() else ""
    tmp_path.unlink(missing_ok=True)

    # Strip ANSI codes
    raw_full = re.sub(r'\x1b\[[0-9;]*[mGKHFABCDJKlh]', '', raw_full)
    raw_full = re.sub(r'\x1b[()][AB012]',               '', raw_full)
    raw_full = re.sub(r'[\r\x00]',                       '', raw_full)

    # Find model output
    model_start = raw_full.rfind("<start_of_turn>model")
    search_text = raw_full[model_start:] if model_start != -1 else raw_full

    # Use finditer, take last match
    matches = list(re.finditer(r'"triage_level"\s*:\s*"([^"]+)"', search_text, re.IGNORECASE))
    m = matches[-1] if matches else None
    level = _normalise(m.group(1)) if m else None

    # Attempt full JSON parse
    result: dict = {"triage_level": level, "latency_s": round(latency, 2)}
    clean = search_text
    if "[End thinking]" in clean:
        clean = clean.split("[End thinking]")[-1].strip()
    clean = re.sub(r'```json\s*', '', clean)
    clean = re.sub(r'```\s*',     '', clean)
    start = clean.find("{")
    if start != -1:
        end = clean.rfind("}") + 1
        js  = clean[start:end] if end > start else clean[start:] + "}"
        js  = re.sub(r",\s*}", "}", js)
        js  = re.sub(r",\s*]", "]", js)
        try:
            parsed = json.loads(js)
            result.update(parsed)
            result["latency_s"] = round(latency, 2)
        except json.JSONDecodeError:
            pass

    return result


print("Inference engine ready. run_triage(text, lang) is available.")

## Benchmark — 10 Clinical Test Cases

The 10 cases below cover all five triage levels (RED, ORANGE, YELLOW, GREEN, BLUE)
across all five benchmark languages (English, Swahili, Tagalog, Hausa, Bengali).
These are a subset of the full 20-case evaluation on which the fine-tuned model achieved
**95% exact match accuracy** and **100% safe escalation rate**.

In [ ]:
# ── Cell 6: Test cases (from scripts/prompt_tuner.py) ────────────────────
TEST_CASES = [
    # RED
    {
        "id": "R1", "lang": "en", "expected": "red",
        "text": "Patient not breathing, no pulse, lips blue. Bystander CPR in progress.",
    },
    {
        "id": "R2", "lang": "sw", "expected": "red",
        "text": "Mtoto wa miaka 3 ana degedege la dakika 8. Hajui chochote. Homa kali.",
    },
    # ORANGE
    {
        "id": "O1", "lang": "en", "expected": "orange",
        "text": (
            "Adult male. Central chest pain 45 min, radiates to left arm. "
            "Sweating. HR 112, RR 22, BP 130/85. Fully alert and talking."
        ),
    },
    {
        "id": "O2", "lang": "tl", "expected": "orange",
        "text": (
            "Babae, 55 taong gulang. Biglaang kahinaan ng kanang braso at "
            "pagbagsak ng mukha, 1 oras na. Nakakapagsalita pa, BP 160/100."
        ),
    },
    # YELLOW
    {
        "id": "Y1", "lang": "ha", "expected": "yellow",
        "text": (
            "Mace shekaru 35. Ciwon kai mai tsanani tun safe. "
            "Babu zazzabi, babu amai, hangen neta daidai. "
            "Tana iya tafiya."
        ),
    },
    {
        "id": "Y2", "lang": "en", "expected": "yellow",
        "text": (
            "Child 4 years. Fever 38.9C for 2 days, alert, crying. "
            "RR 28, HR 118. Eating less but drinking. No seizures."
        ),
    },
    # GREEN
    {
        "id": "G1", "lang": "en", "expected": "green",
        "text": (
            "Minor laceration to left forearm from kitchen knife. "
            "Alert and oriented. HR 80, RR 16, BP 120/78. Bleeding controlled."
        ),
    },
    {   # Bengali script falls outside ASCII, so Python auto-escapes each character
        # as \uXXXX (Unicode code points). The string is identical at runtime —
        # e.g. ২০ == "২০" (20). The text reads: "20-year-old girl.
        # Runny nose for 3 days, mild cough. No fever, no breathing difficulty."
        "id": "G2", "lang": "bn", "expected": "green",
        "text": (
            "২০ বছরের মেয়ে। গত ৩ দিন ধরে নাক দিয়ে পানি পড়ছে, হালকা কাশি। "
            "জ্বর নেই, শ্বাসকষ্ট নেই। খাওয়া দাওয়ন স্বাভাবিক।"
        ),
    },
    # BLUE
    {
        "id": "B1", "lang": "en", "expected": "blue",
        "text": (
            "Patient brought in by family. No breathing, no pulse, "
            "fixed dilated pupils. Body cold to touch. Rigor mortis present. "
            "Family states found at home several hours ago."
        ),
    },
    {
        "id": "B2", "lang": "sw", "expected": "blue",
        "text": (
            "Mzee wa miaka 80 aliyeletwa na familia. Hakuna mapigo ya moyo, "
            "hakuna pumzi. Mwili baridi. Familia inasema alifariki usiku wa jana."
        ),
    },
]

LANG_DISPLAY = {
    "en": "English", "sw": "Swahili", "tl": "Tagalog",
    "ha": "Hausa",   "bn": "Bengali",
}
_LEVELS = ["red", "orange", "yellow", "green", "blue"]

print(f"{len(TEST_CASES)} test cases loaded.")

In [ ]:
# ── Cell 7: Run benchmark ─────────────────────────────────────────────────
import json

results = []

for case in TEST_CASES:
    lang_name = LANG_DISPLAY.get(case["lang"], case["lang"])
    print(f"\n{'='*60}")
    print(f"Case {case['id']}  |  {lang_name}  |  Expected: {case['expected'].upper()}")
    print(f"{'='*60}")
    print(f"Input: {case['text'][:120]}{'...' if len(case['text']) > 120 else ''}")
    print("Running inference...")

    result = run_triage(case["text"], case["lang"])
    level  = result.get("triage_level")

    # Display full JSON
    display = {k: v for k, v in result.items() if k != "latency_s"}
    print("\nModel output:")
    print(json.dumps(display, indent=2))

    # Correctness
    is_correct = (level == case["expected"])
    tick = "\u2713" if is_correct else "\u2717"
    print(f"\nExtracted level : {level.upper() if level else '?'}  {tick}")

    # Safety: over-triage is safe, under-triage is not
    if level in _LEVELS and case["expected"] in _LEVELS:
        is_safe = _LEVELS.index(level) <= _LEVELS.index(case["expected"])
    else:
        is_safe = False
    safe_label = "SAFE (over-triage OK)" if is_safe and not is_correct else ("SAFE" if is_safe else "UNSAFE (under-triage!)")
    print(f"Clinical safety : {safe_label}")
    print(f"Latency         : {result['latency_s']:.2f} s")

    results.append({
        "id": case["id"],
        "lang": case["lang"],
        "expected": case["expected"],
        "predicted": level,
        "correct": is_correct,
        "safe": is_safe,
        "latency_s": result["latency_s"],
    })

# ── Summary ───────────────────────────────────────────────────────────────
correct_n = sum(1 for r in results if r["correct"])
safe_n    = sum(1 for r in results if r["safe"])
total     = len(results)

print(f"\n{'='*60}")
print("BENCHMARK SUMMARY")
print(f"{'='*60}")
print(f"Exact match accuracy : {correct_n}/{total} ({correct_n/total:.0%})")
print(f"Safe escalation rate : {safe_n}/{total} ({safe_n/total:.0%})")
print()

# Per-level breakdown
print("Per-level accuracy:")
for lvl in _LEVELS:
    lvl_cases = [r for r in results if r["expected"] == lvl]
    if not lvl_cases:
        continue
    n_correct = sum(1 for r in lvl_cases if r["correct"])
    preds = [r["predicted"] for r in lvl_cases if not r["correct"]]
    note  = f"  (predicted: {', '.join(str(p) for p in preds)})" if preds else ""
    print(f"  {lvl.upper():<8} {n_correct}/{len(lvl_cases)}{note}")
print(f"{'='*60}")

## Try It Yourself

Edit `patient_intake` and `language` in the cell below, then run it to get a triage result.

**Language codes:** `en` English · `sw` Swahili · `tl` Tagalog · `ha` Hausa · `bn` Bengali  
VoiceBridge can also handle any other language Gemma 4 supports — just set the language code
(e.g. `"fr"` for French, `"hi"` for Hindi, `"am"` for Amharic) and the model will triage in that language.

In [ ]:
# ── Cell 9: Interactive triage ────────────────────────────────────────────
# Edit these two variables and run the cell:
patient_intake = (
    "Male patient, 58 years old. Sudden onset chest tightness and shortness of breath "
    "for the past 20 minutes. Diaphoretic. BP 138/88, HR 105, SpO2 96%. "
    "Alert and conversational."
)
language = "en"

# ── Run ───────────────────────────────────────────────────────────────────
import json
print("Running triage...\n")
result = run_triage(patient_intake, language)

level = result.get("triage_level", "unknown")
print(f"{'='*50}")
print(f"  TRIAGE LEVEL      : {level.upper() if level else 'UNKNOWN'}")
print(f"{'='*50}")

if result.get("primary_complaint"):
    print(f"  Primary complaint : {result['primary_complaint']}")
if result.get("red_flag_indicators"):
    flags = result["red_flag_indicators"]
    if flags:
        print(f"  Red flag indicators:")
        for f in flags:
            print(f"    - {f}")
    else:
        print(f"  Red flag indicators: none")
if result.get("recommended_action"):
    print(f"  Recommended action: {result['recommended_action']}")
if result.get("confidence_score") is not None:
    print(f"  Confidence score  : {result['confidence_score']:.2f}")
print(f"  Latency           : {result['latency_s']:.2f} s")
print(f"{'='*50}")